# RAPPORT DE POSITIONNEMENT — BGFI BANK SÉNÉGAL
### Analyse Concurrentielle du Secteur Bancaire Sénégalais · 2015–2020

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 0 — IMPORTS GLOBAUX + CHEMINS + FONCTIONS UTILITAIRES
# ⚠️ Exécuter EN PREMIER. Toutes les dépendances sont ici.
# ═══════════════════════════════════════════════════════════════════════════

import os
import base64
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # OBLIGATOIRE pour les serveurs / nbconvert
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from IPython.display import display, HTML, Image

warnings.filterwarnings('ignore')

# ── Paramètres globaux ────────────────────────────────────────────────────
TARGET_BANK  = 'BGFI'
ANNEE_DEBUT  = 2015
ANNEE_FIN    = 2020

# Couleurs
C_BGFI       = '#c00000'   # Rouge BGFI (graphiques barres)
C_BGFI_BLUE  = '#004a99'   # Bleu institutionnel BGFI
C_BARRE      = '#5b9bd5'   # Bleu standard barres
C_AUTRE      = 'lightgray' # Gris autres banques
C_OTHERS     = '#5b9bd5'   # Bleu clair autres banques (scatter)
C_BANDEAU    = '#7B3F00'   # Marron bandeau style PPT

# ── Chemins ────────────────────────────────────────────────────────────────
# ⚠️ NE PAS MODIFIER — Chemins validés en local
path_data = r"C:\Users\user\Documents\Data Ingenieur2\Projet_banque\Data\BASE_SENEGAL2.xlsx"
path_logo = r"C:\Users\user\Documents\Data Ingenieur2\Projet_banque\Data\logo_bgfi.jfif"

# ── Fonctions utilitaires ──────────────────────────────────────────────────

def calcul_tcam(v0, v1, n):
    """Taux de Croissance Annuel Moyen entre v0 et v1 sur n années."""
    try:
        if v0 <= 0 or pd.isna(v0) or pd.isna(v1) or n <= 0:
            return 0.0
        return round(((v1 / v0) ** (1 / n) - 1) * 100, 1)
    except:
        return 0.0

def style_ax(ax):
    """Style épuré commun à tous les graphiques."""
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#dddddd')
    ax.spines['bottom'].set_color('#dddddd')
    ax.tick_params(colors='#444444', length=3)
    ax.set_facecolor('white')

def ajouter_logo(fig):
    """Incruste le logo BGFI en haut à droite de la figure Matplotlib."""
    try:
        logo = mpimg.imread(path_logo)
        ax_logo = fig.add_axes([0.85, 0.88, 0.12, 0.12], anchor='NE', zorder=10)
        ax_logo.imshow(logo)
        ax_logo.axis('off')
    except Exception as e:
        print(f"⚠️ Logo non chargé : {e}")

def get_logo_base64():
    """Encode le logo en base64 pour l'intégrer dans le HTML."""
    try:
        with open(path_logo, 'rb') as f:
            return base64.b64encode(f.read()).decode('utf-8')
    except:
        return ''

def sauvegarder_afficher(fig, nom_fichier):
    """Sauvegarde la figure en PNG et l'affiche dans le HTML généré."""
    plt.savefig(nom_fichier, dpi=120, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=nom_fichier))

print('✅ Configuration chargée avec succès.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 1 — CHARGEMENT DES DONNÉES
# ═══════════════════════════════════════════════════════════════════════════

try:
    df = pd.read_excel(path_data, engine='openpyxl')
    df.columns = df.columns.str.strip()
    print('✅ Fichier Excel lu avec succès.')
except Exception as e:
    print(f'❌ Erreur de lecture : {e}')

print(f'✅ Données chargées : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'   Banques : {df["Sigle"].nunique()} | Années : {sorted(df["ANNEE"].unique().tolist())}')

# ── Tables de référence ───────────────────────────────────────────────────
df_2020          = df[df['ANNEE'] == ANNEE_FIN].copy()
total_bilan_2020 = df_2020['BILAN'].sum()
GROUPE_BGFI      = df[df['Sigle'] == TARGET_BANK]['Goupe_Bancaire'].values[0]

print(f'   BGFI groupe : {GROUPE_BGFI}')
print('🎯 Données prêtes.')

---
## Page de Garde

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 2 — PAGE DE GARDE
# ═══════════════════════════════════════════════════════════════════════════

logo_base64 = get_logo_base64()
logo_html   = f'data:image/jpeg;base64,{logo_base64}' if logo_base64 else ''

html_intro = f"""
<div style="font-family: 'Arial', sans-serif; background-color: white;
            padding: 0; border: 1px solid #eee; min-height: 500px;">

    <div style="padding: 40px 0 20px 50px;">
        <h1 style="color: {C_BGFI_BLUE}; font-weight: 800; font-size: 34px;
                   margin: 0; letter-spacing: -1px;">
             STRATÉGIE BGFI BANK
        </h1>
        <p style="color: #666; font-size: 18px; margin-top: 5px;">
            Analyse comparative et dynamique de croissance (2015-2020)
        </p>
    </div>

    <div style="background-color: {C_BGFI_BLUE}; padding: 20px 0; width: 100%;
                margin: 25px 0; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <h2 style="color: white; text-align: center; font-weight: bold; font-size: 24px;
                   margin: 0; text-transform: uppercase; letter-spacing: 3px;">
            Positionnement stratégique &amp; Performance
        </h2>
    </div>

    <div style="padding: 30px 60px; color: #333; line-height: 1.8;">

        <div style="margin-bottom: 25px; display: flex; align-items: flex-start;">
            <span style="color: {C_BGFI_BLUE}; font-size: 25px; margin-right: 15px;">•</span>
            <span><b style="color: {C_BGFI_BLUE}; font-size: 18px;">Contexte :</b><br>
            Étude approfondie des parts de marché et de la rentabilité de
            <b>BGFI Bank Sénégal</b> face à ses principaux concurrents du secteur.</span>
        </div>

        <div style="margin-bottom: 25px; display: flex; align-items: flex-start;">
            <span style="color: {C_BGFI_BLUE}; font-size: 25px; margin-right: 15px;">•</span>
            <span><b style="color: {C_BGFI_BLUE}; font-size: 18px;">Méthodologie :</b><br>
            Analyse basée sur les données financières consolidées 2015-2020,
            utilisant les calculs de TCAM et de ratios de productivité.</span>
        </div>

        <div style="margin-bottom: 25px; display: flex; align-items: flex-start;">
            <span style="color: {C_BGFI_BLUE}; font-size: 25px; margin-right: 15px;">•</span>
            <span><b style="color: {C_BGFI_BLUE}; font-size: 18px;">Objectif :</b><br>
            Mettre en lumière la montée en puissance de la banque sur les segments
            prioritaires (Total Bilan, Emplois, Ressources).</span>
        </div>

    </div>

    <div style="text-align: center; padding: 40px 0; border-top: 1px solid #f5f5f5;">
        {'<img src="' + logo_html + '" alt="Logo BGFI" style="width: 200px; filter: contrast(1.1);">' if logo_html else '<p style="color:#aaa;">Logo BGFI</p>'}
        <div style="width: 60px; height: 4px; background-color: {C_BGFI_BLUE};
                    margin: 20px auto 0;"></div>
    </div>

</div>
"""

display(HTML(html_intro))

---
## 1-1 Total Bilan des Banques — 2020

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 3 — TOTAL BILAN DES BANQUES 2020
# ═══════════════════════════════════════════════════════════════════════════

d_plot     = df_2020.sort_values('BILAN', ascending=True)
nb_banques = len(d_plot)
total_mds  = round(total_bilan_2020 / 1_000, 0)
couleurs   = [C_BGFI if s == TARGET_BANK else C_BARRE for s in d_plot['Sigle']]

fig, ax = plt.subplots(figsize=(14, 9), facecolor='white')
ax.set_facecolor('white')

bars = ax.barh(d_plot['Sigle'], d_plot['BILAN'], color=couleurs, height=0.7)

# Valeurs au bout des barres
max_val = d_plot['BILAN'].max()
for bar in bars:
    width = bar.get_width()
    ax.text(width + (max_val * 0.01), bar.get_y() + bar.get_height() / 2,
            f'{width:,.0f}', va='center', fontsize=9, color='#333333')

# BGFI en gras / rouge
for label in ax.get_yticklabels():
    if label.get_text() == TARGET_BANK:
        label.set_fontweight('bold')
        label.set_color(C_BGFI)

plt.suptitle(
    f'Le secteur Bancaire Sénégalais compte {nb_banques} banques représentant un\n'
    f'total bilan de {total_mds:,.0f} milliards de FCFA soit près de 60% du PIB national.',
    fontsize=14, fontweight='bold', x=0.5, y=0.96
)
ax.set_title(f'Total Bilan des Banques, {ANNEE_FIN} (millions FCFA)',
             fontsize=11, style='italic', color='#555555', pad=15)

ax.spines[['top', 'right', 'bottom']].set_visible(False)
ax.get_xaxis().set_visible(False)
ax.tick_params(axis='y', length=0)
ajouter_logo(fig)

plt.tight_layout(rect=[0, 0.03, 1, 0.90])
sauvegarder_afficher(fig, 'temp_graph_1.png')

---
## 1-2 Un Marché en Croissance — Évolution 2015–2020

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 4 — ÉVOLUTION DU MARCHÉ (4 indicateurs)
# ═══════════════════════════════════════════════════════════════════════════

evolution_marche = df.groupby('ANNEE')[['BILAN', 'EMPLOI', 'RESSOURCES', 'COMPTE']].sum().reset_index()

metrics = [
    ('BILAN',      'Total Bilan',          '#5b9bd5'),
    ('EMPLOI',     'Emplois (Crédits)',     '#70ad47'),
    ('RESSOURCES', 'Ressources (Dépôts)',   '#ed7d31'),
    ('COMPTE',     'Nombre de Comptes',     '#ffc000'),
]

fig, axs = plt.subplots(2, 2, figsize=(14, 8), facecolor='white')

for i, (col, title, color) in enumerate(metrics):
    ax   = axs.flat[i]
    bars = ax.bar(evolution_marche['ANNEE'].astype(str),
                  evolution_marche[col], color=color, alpha=0.7)

    v0 = evolution_marche[evolution_marche['ANNEE'] == ANNEE_DEBUT][col].values[0]
    v1 = evolution_marche[evolution_marche['ANNEE'] == ANNEE_FIN][col].values[0]
    t  = calcul_tcam(v0, v1, ANNEE_FIN - ANNEE_DEBUT)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., h + h * 0.01,
                f'{h:,.0f}', ha='center', va='bottom', fontsize=8)

    ax.set_title(f'{title} (TCAM : +{t}%)', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.get_yaxis().set_visible(False)

ajouter_logo(fig)
plt.tight_layout()
sauvegarder_afficher(fig, 'temp_graph_2.png')

---
## 1-3 Évolution de BGFI Bank — Indicateurs Bilanciels

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 5 — ÉVOLUTION DE BGFI BANK (Bilan + Emplois/Ressources)
# ═══════════════════════════════════════════════════════════════════════════

df_bgfi_evol = df[df['Sigle'] == TARGET_BANK].sort_values('ANNEE')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), facecolor='white')

# Graphique A : Évolution du Total Bilan (Barres)
ax1.bar(df_bgfi_evol['ANNEE'].astype(str), df_bgfi_evol['BILAN'],
        color=C_BGFI_BLUE, alpha=0.8, width=0.6)

for i, val in enumerate(df_bgfi_evol['BILAN']):
    ax1.text(i, val + (val * 0.02), f'{val:,.0f}',
             ha='center', fontweight='bold', color=C_BGFI_BLUE)

ax1.set_title(f'Évolution du Total Bilan ({TARGET_BANK})', fontweight='bold', pad=15)
ax1.set_ylabel('Millions FCFA')
style_ax(ax1)

# Graphique B : Emplois vs Ressources (Lignes)
ax2.plot(df_bgfi_evol['ANNEE'].astype(str), df_bgfi_evol['EMPLOI'],
         marker='o', markersize=8, linewidth=3, color=C_BGFI_BLUE,
         label='Emplois (Crédits)')
ax2.plot(df_bgfi_evol['ANNEE'].astype(str), df_bgfi_evol['RESSOURCES'],
         marker='s', markersize=8, linewidth=3, color='#5b9bd5',
         label='Ressources (Dépôts)', linestyle='--')

last_emp = df_bgfi_evol['EMPLOI'].iloc[-1]
last_res = df_bgfi_evol['RESSOURCES'].iloc[-1]
ax2.text(5, last_emp, f' {last_emp:,.0f}', va='center', fontweight='bold', color=C_BGFI_BLUE)
ax2.text(5, last_res, f' {last_res:,.0f}', va='center', fontweight='bold', color='#5b9bd5')

ax2.set_title(f'Dynamique Emplois vs Ressources ({TARGET_BANK})', fontweight='bold', pad=15)
ax2.legend(frameon=False)
style_ax(ax2)

plt.suptitle(f'Trajectoire de Croissance Interne — {TARGET_BANK} Bank Sénégal',
             fontsize=16, fontweight='bold', y=1.02)

ajouter_logo(fig)
plt.tight_layout()
sauvegarder_afficher(fig, 'temp_graph_evol_bgfi.png')

---
## 1-4 Positionnement Stratégique — BGFI vs Toutes les Banques

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 6 — POSITIONNEMENT BGFI vs TOUTES LES BANQUES (scatter)
# ═══════════════════════════════════════════════════════════════════════════

metrics   = ['BILAN', 'EMPLOI', 'RESSOURCES', 'COMPTE']
titles    = ['Total Bilan', 'Emplois (Crédits)', 'Ressources (Dépôts)', 'Nombre de Comptes']
all_banks = df['Sigle'].unique()

fig, axs = plt.subplots(2, 2, figsize=(16, 12), facecolor='white')

for i, m in enumerate(metrics):
    ax           = axs.flat[i]
    total_m_2020 = df_2020[m].sum()

    for b in all_banks:
        df_b    = df[df['Sigle'] == b].sort_values('ANNEE')
        v_start = df_b[df_b['ANNEE'] == ANNEE_DEBUT][m].values
        v_end   = df_b[df_b['ANNEE'] == ANNEE_FIN][m].values

        if len(v_start) > 0 and len(v_end) > 0:
            ms      = (v_end[0] / total_m_2020) * 100
            growth  = calcul_tcam(v_start[0], v_end[0], 5)
            is_bgfi = (str(b).upper() == TARGET_BANK)

            ax.scatter(ms, growth,
                       color=C_BGFI_BLUE if is_bgfi else C_OTHERS,
                       s=350 if is_bgfi else 120,
                       edgecolor='white', linewidth=1.2,
                       alpha=0.85, zorder=5 if is_bgfi else 3)

            ax.annotate(f'{b}', (ms, growth),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8,
                        fontweight='bold' if is_bgfi else 'normal',
                        color=C_BGFI_BLUE if is_bgfi else '#555555')

    ax.axhline(0, color='#666666', linewidth=1, linestyle='-', alpha=0.2)
    ax.set_title(f'Positionnement : {titles[i]}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Part de Marché 2020 (%)', fontsize=10)
    ax.set_ylabel('TCAM 2015-2020 (%)', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', alpha=0.3)

plt.suptitle(
    f'Cartographie Concurrentielle : {TARGET_BANK} Bank vs Secteur Bancaire\n'
    '(Positionnement Part de Marché vs Dynamique de Croissance)',
    fontsize=16, fontweight='bold', y=0.98, color='#1a1a1a'
)

ajouter_logo(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
sauvegarder_afficher(fig, 'pos_complet_bgfi_bleu.png')

---
## 1-5 Positionnement des Groupes Bancaires

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 7 — POSITIONNEMENT DES GROUPES BANCAIRES (style PPT original)
# ═══════════════════════════════════════════════════════════════════════════

df_gp = df.groupby(['Goupe_Bancaire', 'ANNEE'])[['BILAN', 'EMPLOI', 'RESSOURCES', 'COMPTE']].sum().reset_index()

def scatter_groupes(ax, titre, metric):
    total_m_2020 = df_gp[df_gp['ANNEE'] == ANNEE_FIN][metric].sum()
    groupes_uniq = df_gp['Goupe_Bancaire'].unique()

    for gp in groupes_uniq:
        d_g     = df_gp[df_gp['Goupe_Bancaire'] == gp].sort_values('ANNEE')
        v_start = d_g[d_g['ANNEE'] == ANNEE_DEBUT][metric].values
        v_end   = d_g[d_g['ANNEE'] == ANNEE_FIN][metric].values

        if len(v_start) > 0 and len(v_end) > 0:
            ms         = (v_end[0] / total_m_2020) * 100
            growth     = calcul_tcam(v_start[0], v_end[0], 5)
            is_bgfi_gp = (gp == GROUPE_BGFI)

            ax.scatter(ms, growth,
                       color=C_BGFI_BLUE if is_bgfi_gp else C_OTHERS,
                       s=450 if is_bgfi_gp else 180,
                       edgecolor='white', linewidth=1.5,
                       zorder=5 if is_bgfi_gp else 3)

            if ms > 1 or is_bgfi_gp:
                ax.annotate(f' {gp}', (ms, growth),
                            fontsize=8.5,
                            fontweight='bold' if is_bgfi_gp else 'normal',
                            color=C_BGFI_BLUE if is_bgfi_gp else '#444444',
                            xytext=(5, 5), textcoords='offset points')

    ax.set_title(titre, fontsize=10, fontweight='bold', pad=10)
    ax.set_xlabel('Part de Marché du Groupe (%)', fontsize=8)
    ax.set_ylabel('TCAM (%)', fontsize=8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    style_ax(ax)
    ax.grid(True, linestyle=':', alpha=0.3)

fig, axs = plt.subplots(2, 2, figsize=(15, 11), facecolor='white')

scatter_groupes(axs[0, 0], 'Sur le nombre de comptes', 'COMPTE')
scatter_groupes(axs[0, 1], 'Sur le total bilan',       'BILAN')
scatter_groupes(axs[1, 0], 'Sur les ressources',       'RESSOURCES')
scatter_groupes(axs[1, 1], 'Sur les emplois',          'EMPLOI')

fig.text(0.01, 0.98,
    f'Le positionnement des groupes bancaires confirme la domination des grands groupes\n'
    f'et la montée en puissance du groupe {GROUPE_BGFI}.',
    fontsize=13, fontweight='bold', va='top', color='black')

fig.text(0.5, 0.92,
    'Positionnement des Groupes Bancaires — Marché Sénégalais',
    ha='center', fontsize=11, fontweight='bold', color='white',
    bbox=dict(boxstyle='square,pad=0.6', facecolor=C_BANDEAU, edgecolor='none'))

ajouter_logo(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.90])
sauvegarder_afficher(fig, 'pos_groupes_original.png')

---
## 1-6 Positionnement de BGFI au sein de son Groupe

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 8 — BGFI vs LES BANQUES DE SON GROUPE (style PPT original)
# ═══════════════════════════════════════════════════════════════════════════

df_groupe      = df_2020[df_2020['Goupe_Bancaire'] == GROUPE_BGFI].copy()
banques_groupe = df_groupe['Sigle'].unique()

def scatter_bnk(ax, titre, metric):
    total_m = df_2020[metric].sum()

    for b in banques_groupe:
        df_b    = df[df['Sigle'] == b].sort_values('ANNEE')
        v_start = df_b[df_b['ANNEE'] == ANNEE_DEBUT][metric].values
        v_end   = df_b[df_b['ANNEE'] == ANNEE_FIN][metric].values

        if len(v_start) > 0 and len(v_end) > 0:
            ms      = (v_end[0] / total_m) * 100
            growth  = calcul_tcam(v_start[0], v_end[0], 5)
            is_bgfi = (b == TARGET_BANK)

            ax.scatter(ms, growth,
                       color=C_BGFI_BLUE if is_bgfi else C_OTHERS,
                       s=400 if is_bgfi else 150,
                       edgecolor='white', linewidth=1.5,
                       zorder=5 if is_bgfi else 3)

            ax.annotate(f' {b}', (ms, growth),
                        fontsize=9,
                        fontweight='bold' if is_bgfi else 'normal',
                        color=C_BGFI_BLUE if is_bgfi else '#555555',
                        xytext=(5, 5), textcoords='offset points')

    ax.set_title(titre, fontsize=10, fontweight='bold', pad=10)
    ax.set_xlabel('Part de Marché (%)', fontsize=8)
    ax.set_ylabel('TCAM (%)', fontsize=8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.grid(True, linestyle=':', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

fig, axs = plt.subplots(2, 2, figsize=(15, 11), facecolor='white')

scatter_bnk(axs[0, 0], 'Sur le nombre de comptes', 'COMPTE')
scatter_bnk(axs[0, 1], 'Sur le total bilan',       'BILAN')
scatter_bnk(axs[1, 0], 'Sur les ressources',       'RESSOURCES')
scatter_bnk(axs[1, 1], 'Sur les emplois',          'EMPLOI')

fig.text(0.01, 0.98,
    f'{TARGET_BANK} a une dynamique plus forte et continue de gagner des parts de marché\n'
    'sur tous les indicateurs.',
    fontsize=14, fontweight='bold', va='top', color='black')

fig.text(0.5, 0.92,
    f'Positionnement de {TARGET_BANK} par rapport aux banques du {GROUPE_BGFI}',
    ha='center', fontsize=11, fontweight='bold', color='white',
    bbox=dict(boxstyle='square,pad=0.6', facecolor=C_BANDEAU, edgecolor='none'))

ajouter_logo(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.90])
sauvegarder_afficher(fig, 'pos_groupe_original.png')

---
## 1-7 Productivité Commerciale — BGFI vs Marché

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 9 — PRODUCTIVITÉ BGFI VS MARCHÉ (par agence & par agent)
# ═══════════════════════════════════════════════════════════════════════════

row_bgfi = df_2020[df_2020['Sigle'] == TARGET_BANK].iloc[0]

moy_empl_ag  = (df_2020['EMPLOI']     / df_2020['AGENCE']).replace([np.inf, -np.inf], np.nan).dropna().mean()
moy_ress_ag  = (df_2020['RESSOURCES'] / df_2020['AGENCE']).replace([np.inf, -np.inf], np.nan).dropna().mean()
moy_empl_eff = (df_2020['EMPLOI']     / df_2020['EFFECTIF']).replace([np.inf, -np.inf], np.nan).dropna().mean()
moy_ress_eff = (df_2020['RESSOURCES'] / df_2020['EFFECTIF']).replace([np.inf, -np.inf], np.nan).dropna().mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

x     = np.arange(2)
width = 0.35

# Graphique A : Par Agence
bgfi_ag   = [row_bgfi['EMPLOI'] / row_bgfi['AGENCE'], row_bgfi['RESSOURCES'] / row_bgfi['AGENCE']]
marche_ag = [moy_empl_ag, moy_ress_ag]

ax1.bar(x - width/2, bgfi_ag,   width, label=TARGET_BANK,      color=C_BGFI)
ax1.bar(x + width/2, marche_ag, width, label='Moyenne Marché', color=C_AUTRE)
ax1.set_title('Productivité par Agence (M FCFA)', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(['Emplois / Agence', 'Ressources / Agence'])
ax1.legend()
style_ax(ax1)

# Graphique B : Par Agent
bgfi_eff   = [row_bgfi['EMPLOI'] / row_bgfi['EFFECTIF'], row_bgfi['RESSOURCES'] / row_bgfi['EFFECTIF']]
marche_eff = [moy_empl_eff, moy_ress_eff]

ax2.bar(x - width/2, bgfi_eff,   width, label=TARGET_BANK,      color=C_BGFI)
ax2.bar(x + width/2, marche_eff, width, label='Moyenne Marché', color=C_AUTRE)
ax2.set_title('Productivité par Agent (M FCFA)', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['Emplois / Agent', 'Ressources / Agent'])
ax2.legend()
style_ax(ax2)

plt.suptitle(f'Analyse de la Productivité : {TARGET_BANK} Bank vs Moyenne du Secteur',
             fontsize=15, fontweight='bold', y=1.05)

ajouter_logo(fig)
plt.tight_layout()
sauvegarder_afficher(fig, 'temp_graph_4.png')

---
## 1-8 Ranking Productivité — Toutes les Banques

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 10 — RANKING COMPLET PRODUCTIVITÉ (toutes les banques)
# ═══════════════════════════════════════════════════════════════════════════

df_prod = df_2020.copy()
df_prod['EMP_AG'] = df_prod['EMPLOI']     / df_prod['AGENCE']
df_prod['RES_AG'] = df_prod['RESSOURCES'] / df_prod['AGENCE']
df_prod = df_prod.replace([np.inf, -np.inf], 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10), facecolor='white')

# Emplois / Agence
d1      = df_prod.sort_values('EMP_AG', ascending=True)
colors1 = [C_BGFI_BLUE if str(s).upper() == TARGET_BANK else C_OTHERS for s in d1['Sigle']]
bars1   = ax1.barh(d1['Sigle'], d1['EMP_AG'], color=colors1, height=0.7)

for bar in bars1:
    ax1.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():,.0f}', va='center', fontsize=8, color='#444444')

ax1.set_title('Productivité : Emplois / Agence (M FCFA)', fontweight='bold', pad=15)
ax1.spines[['top', 'right', 'bottom']].set_visible(False)
ax1.get_xaxis().set_visible(False)
ax1.tick_params(axis='y', length=0, labelsize=9)
ax1.set_facecolor('white')

# Ressources / Agence
d2      = df_prod.sort_values('RES_AG', ascending=True)
colors2 = [C_BGFI_BLUE if str(s).upper() == TARGET_BANK else C_OTHERS for s in d2['Sigle']]
bars2   = ax2.barh(d2['Sigle'], d2['RES_AG'], color=colors2, height=0.7)

for bar in bars2:
    ax2.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():,.0f}', va='center', fontsize=8, color='#444444')

ax2.set_title('Productivité : Ressources / Agence (M FCFA)', fontweight='bold', pad=15)
ax2.spines[['top', 'right', 'bottom']].set_visible(False)
ax2.get_xaxis().set_visible(False)
ax2.tick_params(axis='y', length=0, labelsize=9)
ax2.set_facecolor('white')

# BGFI en gras sur les axes Y
for ax in [ax1, ax2]:
    for label in ax.get_yticklabels():
        if TARGET_BANK in label.get_text().upper():
            label.set_fontweight('bold')
            label.set_color(C_BGFI_BLUE)

plt.suptitle(
    f'Efficacité du Réseau Physique : Classement Intégral du Marché\n'
    f'(Comparaison directe de {TARGET_BANK} Bank avec ses {len(df_2020)-1} concurrents)',
    fontsize=16, fontweight='bold', y=0.98
)

ajouter_logo(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
sauvegarder_afficher(fig, 'prod_all_banks_bgfi.png')

---
## 1-9 Solidité et Maîtrise des Risques

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 11 — RISQUE & LEVIER FINANCIER (toutes les banques)
# ═══════════════════════════════════════════════════════════════════════════

df_risk = df_2020.copy()
df_risk['RISQUE_RATIO'] = (df_risk['COÛT.DU.RISQUE'] / df_risk['BILAN']) * 100
df_risk['LEVIER']       = df_risk['BILAN'] / df_risk['FONDS.PROPRE']
df_risk = df_risk.replace([np.inf, -np.inf], 0).fillna(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 11), facecolor='white')

# Coût du Risque
d1      = df_risk.sort_values('RISQUE_RATIO', ascending=True)
colors1 = [C_BGFI_BLUE if str(s).upper() == TARGET_BANK else C_OTHERS for s in d1['Sigle']]
bars1   = ax1.barh(d1['Sigle'], d1['RISQUE_RATIO'], color=colors1, height=0.7)

for bar in bars1:
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():.2f}%', va='center', fontsize=8)

ax1.set_title('Coût Net du Risque / Bilan (%)', fontweight='bold', pad=15)

# Levier Financier
d2      = df_risk.sort_values('LEVIER', ascending=True)
colors2 = [C_BGFI_BLUE if str(s).upper() == TARGET_BANK else C_OTHERS for s in d2['Sigle']]
bars2   = ax2.barh(d2['Sigle'], d2['LEVIER'], color=colors2, height=0.7)

for bar in bars2:
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():.1f}x', va='center', fontsize=8)

ax2.set_title('Levier Financier (Bilan / Fonds Propres)', fontweight='bold', pad=15)

for ax in [ax1, ax2]:
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    ax.get_xaxis().set_visible(False)
    ax.tick_params(axis='y', length=0, labelsize=9)
    ax.set_facecolor('white')
    for label in ax.get_yticklabels():
        if TARGET_BANK in label.get_text().upper():
            label.set_fontweight('bold')
            label.set_color(C_BGFI_BLUE)
            label.set_fontsize(10)

plt.suptitle(
    f'Solidité et Maîtrise des Risques : Classement Complet du Secteur\n'
    f'(Positionnement de {TARGET_BANK} Bank par rapport aux autres banques)',
    fontsize=16, fontweight='bold', y=0.98
)

ajouter_logo(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
sauvegarder_afficher(fig, 'risk_all_banks_bgfi.png')

---
## 1-10 Tableau de Synthèse Stratégique

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 12 — SYNTHÈSE GLOBALE
# ═══════════════════════════════════════════════════════════════════════════

row          = df_2020[df_2020['Sigle'] == TARGET_BANK].iloc[0]
total_m_2020 = df_2020['BILAN'].sum()

data_final = {
    'Indicateur': [
        'Total Bilan',
        'Part de Marché',
        'ROE (Rentabilité)',
        'Coût du Risque',
        'Productivité / Agent',
    ],
    TARGET_BANK: [
        f"{row['BILAN']:,.0f} M",
        f"{(row['BILAN'] / total_m_2020) * 100:.2f}%",
        f"{(row['RESULTAT.NET'] / row['FONDS.PROPRE']) * 100:.1f}%",
        f"{(row['COÛT.DU.RISQUE'] / row['BILAN']) * 100:.2f}%",
        f"{row['EMPLOI'] / row['EFFECTIF']:,.0f} M",
    ],
    'Marché': [
        f"{df_2020['BILAN'].mean():,.0f} M",
        '-',
        f"{(df_2020['RESULTAT.NET'] / df_2020['FONDS.PROPRE']).mean() * 100:.1f}%",
        f"{(df_2020['COÛT.DU.RISQUE'] / df_2020['BILAN']).mean() * 100:.2f}%",
        f"{(df_2020['EMPLOI'] / df_2020['EFFECTIF']).mean():,.0f} M",
    ],
    'Statut': [
        '⚡ Croissance',
        '⚠️ À renforcer',
        '✅ Performant',
        '✅ Maîtrisé',
        '✅ Efficace',
    ],
}

df_final   = pd.DataFrame(data_final)
html_table = df_final.to_html(index=False, classes='report-table')

style = f"""
<style>
    .report-table {{
        font-family: Arial, sans-serif;
        border-collapse: collapse;
        width: 100%;
    }}
    .report-table td, .report-table th {{
        border: 1px solid #ddd;
        padding: 12px;
        text-align: center;
    }}
    .report-table th {{
        background-color: {C_BGFI_BLUE};
        color: white;
        font-weight: bold;
    }}
    .report-table tr:nth-child(even) {{ background-color: #f9f9f9; }}
</style>
"""

print('\n' + '='*50 + '\n  TABLEAU DE SYNTHÈSE STRATÉGIQUE\n' + '='*50)
display(HTML(style + html_table))